# Comparing Transformations [Step 5 - Which One, When, and at What Cost]

> **MLCourse - Agentic AI - Advanced RAG - Query Transformation**

Four techniques, one evaluation set. This notebook runs the baseline, HyDE,
multi-query expansion and step-back prompting over the same questions with the
same relevance rules, and reports precision@k, MRR, latency and LLM call count
for each.

As in [`../11_reranking/04_measuring_the_lift.ipynb`](../11_reranking/04_measuring_the_lift.ipynb),
we report the **real** numbers, including where a technique does nothing or
makes things worse. That outcome is information, not failure.

### 1. Setup


In [1]:
import os
import re
import time
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
from dotenv import load_dotenv


def find_env(start=None):
    """Walk up from the notebook directory until a .env file appears."""
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        candidate = folder / ".env"
        if candidate.exists():
            return candidate
    raise FileNotFoundError("No .env found walking up from " + str(start))


ENV_PATH = find_env()
load_dotenv(ENV_PATH)
DATA_DIR = ENV_PATH.parent / "data"

print("env file :", ENV_PATH)
print("data dir :", DATA_DIR)
print("GROQ_API_KEY present:", bool(os.environ.get("GROQ_API_KEY")))

env file : D:\projects\python\MLCourse\03_agentic_ai\.env
data dir : D:\projects\python\MLCourse\03_agentic_ai\data
GROQ_API_KEY present: True


In [2]:
from langchain_groq import ChatGroq

GROQ_MODEL = "qwen/qwen3.8-27b"          # verified available on this account
llm = ChatGroq(model=GROQ_MODEL, temperature=0)

THINK_RE = re.compile(r"<think>.*?</think>", re.DOTALL)


def clean(text):
    """Strip any <think>...</think> block a reasoning model may emit."""
    return THINK_RE.sub("", text).strip()


def ask(prompt, retries=4, pause=1.5):
    """Call Groq with exponential backoff. Free tier is roughly 8000 tokens/minute,
    so every loop in these notebooks paces itself and retries on rate limits."""
    delay = 5.0
    for attempt in range(retries):
        try:
            answer = clean(llm.invoke(prompt).content)
            time.sleep(pause)
            return answer
        except Exception as exc:
            if attempt == retries - 1:
                raise
            print(f"  [retry {attempt + 1}] {type(exc).__name__} - sleeping {delay:.0f}s")
            time.sleep(delay)
            delay *= 2


print("Groq model:", GROQ_MODEL)
print("smoke test:", ask("Reply with exactly one word: ready"))

Groq model: qwen/qwen3.8-27b


smoke test: ready


In [3]:
ALICE_PATH = DATA_DIR / "alice.txt"
raw_text = ALICE_PATH.read_text(encoding="utf-8-sig")

# Paragraph-sized chunks: human-readable units, good enough for retrieval demos
# and identical to the chunking used in ../01_hybrid_search.
paragraphs = [" ".join(p.split()) for p in raw_text.split("\n\n") if len(p.strip()) > 200]

print("characters :", len(raw_text))
print("paragraphs :", len(paragraphs))
print("example    :", paragraphs[10][:150], "...")

characters : 144696
paragraphs : 237
example    : Alice was not a bit hurt, and she jumped up on to her feet in a moment: she looked up, but it was all dark overhead; before her was another long passa ...


In [4]:
from sentence_transformers import SentenceTransformer
import numpy as np

encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
doc_vectors = encoder.encode(paragraphs, normalize_embeddings=True,
                             batch_size=64, show_progress_bar=False)


def embed(text):
    return encoder.encode([text], normalize_embeddings=True)[0]


def dense_rank(text, top_n=10):
    """Rank paragraph indices by cosine similarity to `text` (best first)."""
    sims = doc_vectors @ embed(text)
    return [int(i) for i in np.argsort(sims)[::-1][:top_n]]


def dense_scores(text):
    return doc_vectors @ embed(text)


print("dense index ready:", doc_vectors.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

dense index ready: (237, 384)


In [5]:
# An evaluation question is only useful if we can decide, mechanically and
# without an LLM, whether a retrieved paragraph is relevant. We do that with
# required keyword sets: a paragraph counts as relevant when it contains every
# keyword in at least one of the "any_of" groups. This is a strict, honest,
# reproducible judgement - no LLM grading, no hand-waving.

EVAL_QUESTIONS = [
    {"q": "Why was the White Rabbit in such a hurry?",
     "any_of": [["rabbit", "hurry"], ["rabbit", "late"], ["oh dear", "late"]]},
    {"q": "What happened when Alice drank from the little bottle?",
     "any_of": [["drink", "bottle"], ["bottle", "shutting up like a telescope"],
                ["drank", "telescope"]]},
    {"q": "What game does the Queen of Hearts make everyone play?",
     "any_of": [["croquet"], ["flamingo", "hedgehog"]]},
    {"q": "Who does Alice meet at the mad tea party?",
     "any_of": [["hatter", "dormouse"], ["march hare", "hatter"], ["tea", "dormouse"]]},
    {"q": "What advice does the Caterpillar give Alice?",
     "any_of": [["caterpillar", "mushroom"], ["caterpillar", "keep your temper"],
                ["caterpillar", "who are you"]]},
    {"q": "How does the Cheshire Cat disappear?",
     "any_of": [["grin", "vanish"], ["cheshire cat", "grin"], ["vanished", "grin"]]},
    {"q": "What does the Queen shout whenever she is angry?",
     "any_of": [["off with"], ["queen", "executed"]]},
    {"q": "What happens at the trial of the Knave of Hearts?",
     "any_of": [["knave", "tarts"], ["jury", "verdict"], ["sentence", "verdict"]]},
]


def is_relevant(doc_text, question):
    """True when the paragraph satisfies any one keyword group for the question."""
    low = doc_text.lower()
    return any(all(word in low for word in group) for group in question["any_of"])


def precision_at_k(ranked_ids, question, k=5):
    """Fraction of the top-k retrieved paragraphs that are relevant."""
    top = ranked_ids[:k]
    return sum(is_relevant(paragraphs[i], question) for i in top) / max(len(top), 1)


# Sanity check: every question must have at least one relevant paragraph in
# the corpus, otherwise the metric is meaningless.
for question in EVAL_QUESTIONS:
    n_rel = sum(is_relevant(p, question) for p in paragraphs)
    print(f"{n_rel:3d} relevant paragraphs | {question['q']}")

  7 relevant paragraphs | Why was the White Rabbit in such a hurry?
  3 relevant paragraphs | What happened when Alice drank from the little bottle?
  9 relevant paragraphs | What game does the Queen of Hearts make everyone play?
 10 relevant paragraphs | Who does Alice meet at the mad tea party?
  2 relevant paragraphs | What advice does the Caterpillar give Alice?
  1 relevant paragraphs | How does the Cheshire Cat disappear?
  5 relevant paragraphs | What does the Queen shout whenever she is angry?
  1 relevant paragraphs | What happens at the trial of the Knave of Hearts?


### 2. The four strategies, side by side

Each strategy is a function `query -> ranked document ids`, so they are
interchangeable and the comparison is apples to apples. We also count LLM calls,
because that is the real cost driver on a rate-limited free tier.

In [6]:
import numpy as np

CALL_COUNT = {"n": 0}
_raw_ask = ask


def ask(prompt, **kwargs):            # noqa: F811 - wrap to count calls
    CALL_COUNT["n"] += 1
    return _raw_ask(prompt, **kwargs)


def rrf_fuse(rankings, k=60, top_n=20):
    scores = {}
    for ranked in rankings:
        for rank, doc_id in enumerate(ranked, 1):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank)
    return [d for d, _ in sorted(scores.items(), key=lambda kv: kv[1],
                                 reverse=True)[:top_n]]


def s_baseline(question):
    """No transformation - embed the question as typed."""
    return dense_rank(question, top_n=20)


def s_hyde(question):
    """Generate a hypothetical answer passage and embed that instead."""
    doc = ask(
        "Write a short passage (about 70 words) from Lewis Carroll's 'Alice's "
        "Adventures in Wonderland' that answers the question below. Write it in "
        "the novel's narrative style. Do not hedge and do not mention the "
        f"question.\n\nQuestion: {question}\n\nPassage:"
    )
    return dense_rank(doc, top_n=20)


def s_multi_query(question, n=3):
    """Generate rephrasings, retrieve with each, fuse the ranks with RRF."""
    raw = ask(
        f"Generate {n} alternative search queries for the question below. Vary "
        "the vocabulary, specificity and grammatical form. One per line, no "
        f"numbering, nothing else.\n\nQuestion: {question}"
    )
    variants = [re.sub(r"^[\-\d\.\)\s]+", "", ln).strip()
                for ln in raw.splitlines() if ln.strip()][:n]
    return rrf_fuse([dense_rank(q, top_n=20) for q in [question] + variants])


def s_step_back(question):
    """Retrieve for the question and for a broader step-back version, then fuse."""
    general = ask(
        "Turn this specific question into a broader 'step-back' question about "
        "the general scene or concept behind it. Keep the characters and "
        "setting. Output only the question.\n\n"
        f"Specific question: {question}\n\nStep-back question:"
    ).strip().strip('"')
    return rrf_fuse([dense_rank(question, top_n=20), dense_rank(general, top_n=20)])


STRATEGIES = {
    "baseline": s_baseline,
    "hyde": s_hyde,
    "multi-query": s_multi_query,
    "step-back": s_step_back,
}
print("strategies:", list(STRATEGIES))

strategies: ['baseline', 'hyde', 'multi-query', 'step-back']


### 3. Run the comparison

Eight questions times four strategies. Three of the strategies make LLM calls,
so we pace the loop - the Groq free tier is roughly 8000 tokens per minute, and
this loop is exactly the kind of thing that trips it.

In [7]:
def reciprocal_rank(ranked_ids, question, limit=20):
    for pos, doc_id in enumerate(ranked_ids[:limit], 1):
        if is_relevant(paragraphs[doc_id], question):
            return 1.0 / pos
    return 0.0


K_VALUES = [1, 3, 5]
records = {name: {"p": {k: [] for k in K_VALUES}, "rr": [], "ms": [], "calls": 0}
           for name in STRATEGIES}

for question in EVAL_QUESTIONS:
    for name, fn in STRATEGIES.items():
        before = CALL_COUNT["n"]
        t0 = time.time()
        ranked = fn(question["q"])
        elapsed_ms = 1000 * (time.time() - t0)

        records[name]["ms"].append(elapsed_ms)
        records[name]["calls"] += CALL_COUNT["n"] - before
        records[name]["rr"].append(reciprocal_rank(ranked, question))
        for k in K_VALUES:
            records[name]["p"][k].append(precision_at_k(ranked, question, k))
    print(f"  done: {question['q'][:55]}")
    time.sleep(3.0)             # pacing for the free-tier token budget

print("\ntotal LLM calls:", CALL_COUNT["n"])

  done: Why was the White Rabbit in such a hurry?


  done: What happened when Alice drank from the little bottle?


  done: What game does the Queen of Hearts make everyone play?


  done: Who does Alice meet at the mad tea party?


  done: What advice does the Caterpillar give Alice?


  done: How does the Cheshire Cat disappear?


  done: What does the Queen shout whenever she is angry?


  done: What happens at the trial of the Knave of Hearts?



total LLM calls: 24


In [8]:
print(f"{'strategy':<14}" + "".join(f"{'P@' + str(k):>8}" for k in K_VALUES)
      + f"{'MRR':>8}{'ms/query':>11}{'LLM calls':>11}")
print("-" * 70)

summary = {}
for name in STRATEGIES:
    r = records[name]
    summary[name] = {
        **{f"P@{k}": float(np.mean(r["p"][k])) for k in K_VALUES},
        "MRR": float(np.mean(r["rr"])),
        "ms": float(np.mean(r["ms"])),
        "calls_per_query": r["calls"] / len(EVAL_QUESTIONS),
    }
    s = summary[name]
    print(f"{name:<14}" + "".join(f"{s['P@' + str(k)]:>8.3f}" for k in K_VALUES)
          + f"{s['MRR']:>8.3f}{s['ms']:>11.0f}{s['calls_per_query']:>11.1f}")

strategy           P@1     P@3     P@5     MRR   ms/query  LLM calls
----------------------------------------------------------------------
baseline         0.500   0.333   0.250   0.681         44        0.0
hyde             0.750   0.542   0.400   0.792       1908        1.0
multi-query      0.375   0.375   0.275   0.650       2072        1.0
step-back        0.625   0.292   0.325   0.744       2070        1.0


In [9]:
base = summary["baseline"]
print("change versus the untransformed baseline\n")
print(f"{'strategy':<14}{'dP@5':>9}{'dMRR':>9}{'extra ms':>11}")
print("-" * 44)
for name in STRATEGIES:
    if name == "baseline":
        continue
    s = summary[name]
    print(f"{name:<14}{s['P@5'] - base['P@5']:>+9.3f}{s['MRR'] - base['MRR']:>+9.3f}"
          f"{s['ms'] - base['ms']:>+11.0f}")

change versus the untransformed baseline

strategy           dP@5     dMRR   extra ms
--------------------------------------------
hyde             +0.150   +0.110      +1864
multi-query      +0.025   -0.031      +2028
step-back        +0.075   +0.062      +2026


### 4. Per-question view

Averages across eight questions hide the interesting behaviour. Transformations
tend to be **high variance**: a big win on one question, a big loss on another.

In [10]:
K = 5
print(f"precision@{K} per question\n")
header = f"{'question':<40}" + "".join(f"{n[:11]:>12}" for n in STRATEGIES)
print(header)
print("-" * len(header))
for i, question in enumerate(EVAL_QUESTIONS):
    row = "".join(f"{records[n]['p'][K][i]:>12.2f}" for n in STRATEGIES)
    print(f"{question['q'][:38]:<40}{row}")

precision@5 per question

question                                    baseline        hyde multi-query   step-back
----------------------------------------------------------------------------------------
Why was the White Rabbit in such a hur          0.40        0.60        0.40        0.60
What happened when Alice drank from th          0.40        0.20        0.20        0.20
What game does the Queen of Hearts mak          0.20        0.80        0.20        0.20
Who does Alice meet at the mad tea par          0.20        0.80        0.60        0.60
What advice does the Caterpillar give           0.20        0.00        0.20        0.20
How does the Cheshire Cat disappear?            0.20        0.20        0.20        0.20
What does the Queen shout whenever she          0.20        0.40        0.20        0.40
What happens at the trial of the Knave          0.20        0.20        0.20        0.20


In [11]:
print("win / tie / loss versus baseline (precision@5)\n")
for name in STRATEGIES:
    if name == "baseline":
        continue
    w = t = l = 0
    for i in range(len(EVAL_QUESTIONS)):
        d = records[name]["p"][K][i] - records["baseline"]["p"][K][i]
        w += d > 1e-9
        l += d < -1e-9
        t += abs(d) <= 1e-9
    print(f"  {name:<14} wins {w}  ties {t}  losses {l}")

win / tie / loss versus baseline (precision@5)

  hyde           wins 4  ties 2  losses 2
  multi-query    wins 1  ties 6  losses 1
  step-back      wins 3  ties 4  losses 1


### 5. Let the model read the scoreboard

The numbers are already fixed; the model's job here is only to turn them into an
engineering recommendation.

In [12]:
table = "\n".join(
    f"{name}: P@1={s['P@1']:.3f} P@3={s['P@3']:.3f} P@5={s['P@5']:.3f} "
    f"MRR={s['MRR']:.3f} latency={s['ms']:.0f}ms llm_calls={s['calls_per_query']:.1f}"
    for name, s in summary.items()
)

print(table)
print("\n--- recommendation ---")
print(ask(
    "You are a retrieval engineer. Below are measured results for four query "
    "strategies on a 237-paragraph corpus with 8 evaluation questions. In 5-7 "
    "sentences, recommend which strategy to ship and say plainly whether the "
    "quality differences justify the extra latency and LLM calls. If a "
    "transformation did not help, say so directly rather than defending it.\n\n"
    + table
))

baseline: P@1=0.500 P@3=0.333 P@5=0.250 MRR=0.681 latency=44ms llm_calls=0.0
hyde: P@1=0.750 P@3=0.542 P@5=0.400 MRR=0.792 latency=1908ms llm_calls=1.0
multi-query: P@1=0.375 P@3=0.375 P@5=0.275 MRR=0.650 latency=2072ms llm_calls=1.0
step-back: P@1=0.625 P@3=0.292 P@5=0.325 MRR=0.744 latency=2070ms llm_calls=1.0

--- recommendation ---


Ship the **hyde** strategy, as it is the only variant that significantly outperforms the baseline across all precision metrics and MRR. The jump in P@1 from 0.500 to 0.750 and MRR from 0.681 to 0.792 represents a substantial quality improvement that justifies the operational overhead. While the latency increases from 44ms to 1908ms and requires one LLM call, this trade-off is acceptable given the nearly 50% relative gain in top-1 accuracy. In contrast, the **multi-query** strategy should be discarded immediately, as it performs worse than the baseline on every metric while incurring similar latency and LLM costs. The **step-back** approach also fails to justify its cost, offering only marginal MRR improvements over the baseline while suffering from a significant drop in P@3 compared to hyde. Therefore, the quality differences for hyde are clear and decisive, whereas the other transformations did not help and should not be deployed.


### 6. A decision table

Independent of what this particular run measured, here is how the four
techniques map to problems - use it to pick a starting point, then measure.

| symptom | technique | why |
|---|---|---|
| retrieved chunks are on-topic but never answer the question | **HyDE** | closes the question/answer asymmetry |
| results are erratic - small rephrasings change everything | **multi-query** | averages away phrasing luck |
| the question asks about a detail inside a bigger process | **step-back** | pulls in the explaining context |
| the question has several independent parts | **decomposition** ([`../03_agentic_rag/03_query_decomposition.ipynb`](../03_agentic_rag/03_query_decomposition.ipynb)) | one retrieval per part |
| the right chunk is retrieved but ranked 8th | **reranking** ([`../11_reranking`](../11_reranking/README.md)) | this is not a query problem at all |

That last row is the most important one to internalise. Query transformation
fixes **recall** - the right document is not in the candidate set. Reranking
fixes **precision** - it is in the set but ranked too low. Diagnose which one you
have before reaching for either, or you will spend LLM calls solving the wrong
problem.

### 7. Composition

These techniques stack, and the standard production stack is:

```
   query
     |
     +-- transform (HyDE / multi-query / step-back)  <- this module
     |
     +-- retrieve wide (BM25 + dense -> RRF, 50 candidates)  <- ../01_hybrid_search
     |
     +-- rerank (cross-encoder -> top 5)  <- ../11_reranking
     |
     +-- generate
```

Each layer adds latency and can be measured independently. Add them one at a
time, measuring after each, and stop when the marginal gain stops paying for
itself.

### 8. Key takeaways

- Compare transformations on **one fixed question set with one relevance rule**,
  or the numbers mean nothing.
- Report latency and LLM calls next to quality - a `+0.02` P@5 for `+800 ms` and
  two extra calls is usually a bad trade.
- Transformations are **high variance**: check per-question wins and losses, not
  just the mean.
- Query transformation fixes recall; reranking fixes precision. Diagnose first.

Next module: [`../13_contextual_retrieval`](../13_contextual_retrieval/README.md)
- what if the problem is not the query or the ranking, but the *chunk*?